In [1]:
# silence warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

In [2]:
%pip install opencv-python
%pip install keras_cv

  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl (72.9 MB)
Note: you may need to restart the kernel to use updated packages.
  Using cached keras_cv-0.9.0-py3-none-any.whl.metadata (12 kB)
  Using cached regex-2026.2.28-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tensorflow_datasets-4.9.9-py3-none-any.whl.metadata (11 kB)
  Using cached keras_core-0.1.7-py3-none-any.whl.metadata (4.3 kB)
  Using cached kagglehub-1.0.0-py3-none-any.whl.metadata (40 kB)
  Using cached kagglesdk-0.1.16-py3-none-any.whl.metadata (13 kB)
  Using cached dm_tree-0.1.9-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.4 kB)
  Using cached array_record-0.8.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.1 kB)
  Using cached etils-1.13.0-py3-none-any.whl.metadata (6.5 kB)
  Using cache

In [3]:
import time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import keras
import keras_cv
from keras.callbacks import ModelCheckpoint
from keras_cv import visualization

In [4]:
# ensure access to GPU
import tensorflow as tf
tf.config.list_physical_devices('GPU')

[]

In [5]:
# pre-trained backbone
backbone = keras_cv.models.ResNet50Backbone.from_preset("resnet50_imagenet")

# retinanet for 3 elephant seal classes
model = keras_cv.models.RetinaNet(
    num_classes=3, 
    backbone=backbone,
    bounding_box_format="xywh", # Your data format
    anchor_generator=keras_cv.models.RetinaNet.default_anchor_generator(
        bounding_box_format="xywh"
    ),
    prediction_decoder=keras_cv.layers.NonMaxSuppression(
        bounding_box_format="xywh",
        from_logits=True, # train with logits
        confidence_threshold=0.2, # lower during training/debugging; bump up to 0.5 later
        iou_threshold=0.5,
    )
)

In [6]:
# map CSV class names to IDs
class_mapping = {'bull': 0, 'cow': 1, 'pup': 2}

def load_dataset(csv_path, img_dir):
    df = pd.read_csv(csv_path, names=['file', 'x1', 'y1', 'x2', 'y2', 'label', 'extra'])
    
    # group by filename to get all boxes for one image together
    grouped = df.groupby('file')
    
    image_paths = []
    all_boxes = []
    all_classes = []
    
    for file, group in grouped:
        image_paths.append(f"{img_dir}/{file}")
        # x1, y1, x2, y2 -> x, y, w, h
        boxes = group[['x1', 'y1', 'x2', 'y2']].values
        boxes[:, 2] = boxes[:, 2] - boxes[:, 0] # width = x2 - x1
        boxes[:, 3] = boxes[:, 3] - boxes[:, 1] # height = y2 - y1
        
        all_boxes.append(boxes.astype('float32'))
        all_classes.append(group['label'].map(class_mapping).values.astype('float32'))

    # Create a TF Dataset
    def load_image(path, boxes, classes):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.cast(img, tf.float32)
        
        # ensure both are explicitly tensors so the dictionary is consistent
        return {
            "images": img, 
            "bounding_boxes": {
                "boxes": tf.cast(boxes, tf.float32), 
                "classes": tf.cast(classes, tf.float32)
            }
        }

    dataset = tf.data.Dataset.from_generator(
        lambda: zip(image_paths, all_boxes, all_classes),
        output_signature=(
            tf.TensorSpec(shape=(), dtype=tf.string),
            tf.TensorSpec(shape=(None, 4), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.float32),
        )
    ).map(load_image)
    
    return dataset

# helper to turn Ragged boxes into Dense (padded) boxes
def pad_to_dense(data):
    boxes = data["bounding_boxes"]["boxes"]
    classes = data["bounding_boxes"]["classes"]
    
    # 1. Convert RaggedTensors to standard Tensors if they are ragged
    if isinstance(boxes, tf.RaggedTensor):
        boxes = boxes.to_tensor(default_value=-1.0)
    if isinstance(classes, tf.RaggedTensor):
        classes = classes.to_tensor(default_value=-1.0)

    # 2. Determine current number of boxes
    # We use tf.shape(boxes)[0] to handle the dynamic dimension
    num_boxes = tf.shape(boxes)[0]
    pad_len = tf.maximum(0, 32 - num_boxes)
    
    # 3. Pad boxes to (32, 4)
    padded_boxes = tf.pad(boxes, [[0, pad_len], [0, 0]], constant_values=-1.0)
    padded_boxes = padded_boxes[:32, :] # Ensure exactly 32
    
    # 4. Pad classes to (32,)
    # We ensure classes is 1D before padding
    classes = tf.reshape(classes, [-1])
    padded_classes = tf.pad(classes, [[0, pad_len]], constant_values=-1.0)
    padded_classes = padded_classes[:32] # Ensure exactly 32
    
    # 5. Set static shapes so the Batching layer is happy
    padded_boxes.set_shape([32, 4])
    padded_classes.set_shape([32])
    
    data["bounding_boxes"] = {
        "boxes": padded_boxes,
        "classes": padded_classes
    }
    return data

# map the dataset to the format the model expects: (input_dict, target_dict)
def prepare_for_model(data):
    # x: The raw image tensor (RetinaNet extracts features from this)
    # y: The bounding box dictionary (LabelEncoder uses this to create targets)
    return data["images"], data["bounding_boxes"]

train_csv = 'data/train/annotations_final_no.csv'
valid_csv = 'data/valid/annotations_final_no.csv'
train_dir = 'data/train'
valid_dir = 'data/valid'

train_ds = load_dataset(train_csv, train_dir)
val_ds = load_dataset(valid_csv, valid_dir)

# resize, handling RaggedTensors correctly
resizing = keras_cv.layers.Resizing(
    640, 640, pad_to_aspect_ratio=True, bounding_box_format="xywh"
)

train_ds = (
    train_ds
    .map(resizing, num_parallel_calls=tf.data.AUTOTUNE)
    .map(pad_to_dense, num_parallel_calls=tf.data.AUTOTUNE) # Pad boxes here!
    .batch(8) # Standard batching now works
    .map(prepare_for_model, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(resizing, num_parallel_calls=tf.data.AUTOTUNE)
    .map(pad_to_dense, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(8)
    .map(prepare_for_model, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# save the above into a file for later use
train_ds.save("data/updated-train-ds", compression="GZIP")
val_ds.save("data/updated-valid-ds", compression="GZIP")

In [ ]:
# # LOAD: This replaces all the CSV, Padding, and Resizing code
# train_ds = tf.data.Dataset.load("data/updated-train-ds")
# val_ds = tf.data.Dataset.load("data/updated-valid-ds")

# # (Optional) Re-apply prefetch for performance
# train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
# val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

In [7]:
# folder for checkpoints
checkpoint_dir = "data/model-checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Define the callback
checkpoint_callback = ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, "retinanet_e_seal_epoch_{epoch:02d}.weights.h5"),
    save_weights_only=True,
    monitor="val_loss",
    mode="min",
    save_best_only=False,
    verbose=1
)

In [ ]:
# Use a small learning rate for fine-tuning
optimizer = keras.optimizers.Adam(learning_rate=1e-4)

model.compile(
    classification_loss="focal",
    box_loss=keras_cv.losses.SmoothL1Loss(), 
    optimizer=keras.optimizers.Adam(learning_rate=1e-4)
)

model.fit(
    train_ds, 
    epochs=5, 
    validation_data=val_ds,
    callbacks=[checkpoint_callback]
)

Epoch 1/5


/opt/conda/lib/python3.13/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'images': 'images'}
Received: inputs=Tensor(shape=(None, 640, 640, 3))
  warnings.warn(msg)
2026-03-03 01:11:22.161655: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-03 01:11:22.320931: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-03 01:11:24.021259: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-03 01:11:24.319991: E external/

    341/Unknown 416s 903ms/step - box_loss: 1.1376e-06 - classification_loss: 0.4342 - loss: 0.4342

2026-03-03 01:17:28.546250: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-03 01:17:28.698667: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-03 01:17:29.130823: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-03 01:17:29.278579: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-03 01:17:31.145326: E external/local_xla/xla/stream_

    342/Unknown 450s 1s/step - box_loss: 1.1376e-06 - classification_loss: 0.4339 - loss: 0.4339   

/opt/conda/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: saving model to ../../Data/elephant-seals/model-checkpoints/retinanet_e_seal_epoch_01.weights.h5

Epoch 1: finished saving model to ../../Data/elephant-seals/model-checkpoints/retinanet_e_seal_epoch_01.weights.h5
342/342 ━━━━━━━━━━━━━━━━━━━━ 481s 1s/step - box_loss: 1.1293e-06 - classification_loss: 0.3294 - loss: 0.3295 - val_box_loss: 1.2432e-06 - val_classification_loss: 0.9357 - val_loss: 0.9214
Epoch 2/5
342/342 ━━━━━━━━━━━━━━━━━━━━ 0s 920ms/step - box_loss: 9.6148e-07 - classification_loss: 0.2072 - loss: 0.2072
Epoch 2: saving model to ../../Data/elephant-seals/model-checkpoints/retinanet_e_seal_epoch_02.weights.h5

Epoch 2: finished saving model to ../../Data/elephant-seals/model-checkpoints/retinanet_e_seal_epoch_02.weights.h5
342/342 ━━━━━━━━━━━━━━━━━━━━ 332s 971ms/step - box_loss: 9.4398e-07 - classification_loss: 0.1853 - loss: 0.1855 - val_box_loss: 1.0610e-06 - val_classification_loss: 2.7228 - val_loss: 2.7599
Epoch 3/5
342/342 ━━━━━━━━━━━━━━━━━━━━ 0s 933ms/ste

In [ ]:
random_image_list = [
    '5MSL3749-174_png.rf.1ade9f0908936fd853dc2ae9a0f00233.jpg',
     '5MSL3764-21_png.rf.d67ea7373c5b15df832b5e11d4d59192.jpg',
     '5MSL3819-65_png.rf.7cf3c5254e14e6b5cf5905c04ef3f848.jpg',
     '5MSL0109-125_png.rf.0999b3e2f7414ce671eb6ece5f074af1.jpg',
     '5MSL0109-4_png.rf.4c177bee4ff3cdfb044448de4af8f62c.jpg',
     '5MSL0087-138_png.rf.6843d941557d06ab666d79fe30071175.jpg',
     '5MSL4416-99_png.rf.1e915260973f7fd233ef130506a02836.jpg',
     '4MSL0120-163_png.rf.fec5aded76dfdbf287db0a33cffecb65.jpg',
     '5MSL3749-24_png.rf.c0d599cb5e1a10d9bb787cbdbef5e201.jpg',
     '5MSL1479-6_png.rf.c9a1b45ca0cc3abd059facca39256aff.jpg',
     '5MSL3749-146_png.rf.adb2315a6cc0e239980419d05ae2fcab.jpg',
     '4MSL0119-104_png.rf.0f73c19f543a90b15c12dc72bf972446.jpg',
     '5MSL0087-120_png.rf.ea046df0b3121de73453abcfc2f6fcb6.jpg',
     '5MSL0109-84_png.rf.e1d78f59110465473cd83883edd58ab0.jpg',
     '5MSL3436-6_png.rf.4ea434812bb5b4e80c37b85c5103168d.jpg',
     '5MSL0095-98_png.rf.d0a2d706df7b447719994fd7de1f7d36.jpg',
     '5MSL0087-22_png.rf.b500d2b83eb6a2b9d76fcd6cbe9c55af.jpg',
     '5MSL0109-95_png.rf.73ae554e192ba0a1cb271ec123f8330f.jpg',
     '5MSL3436-15_png.rf.10de22d575068c5c4d52c255e5c56cfa.jpg',
     '4MSL0120-169_png.rf.374080801c483f31562d9e265eb66652.jpg',
     'MA263076-124_png.rf.42e9d6beed2e154f9051414a40aae592.jpg',
     '5MSL0089-142_png.rf.00c837be2517e38115a9bcd28e1acb21.jpg',
     '5MSL3764-132_png.rf.eaaa0fa497455e8c42fb6698f4f1f125.jpg',
     '5MSL3436-115_png.rf.d2ebe4ef97473624b43c17e3373c338d.jpg',
     '5MSL3764-18_png.rf.7dd55fc701e3f0ec09ef893668bbb45d.jpg',
     '5MSL3764-134_png.rf.294fa1f2ec3227e20995494de77ac3f4.jpg',
     '5MSL3764-76_png.rf.2bfc3f67fdfefc7715222d7aad08ae9c.jpg',
     '4MSL0120-140_png.rf.03e7f0b7cf647b4b8793a1fbaa15844e.jpg',
     '5MSL3819-55_png.rf.a7ec0f8019e2d7ee1d781867ea416e53.jpg',
     '5MSL0087-112_png.rf.d74dfca1a15442986c922680263769f5.jpg',
     '5MSL0019-47_png.rf.abd556ad1262d88c05f035864ab95be2.jpg',
     'MA262988-171_png.rf.6d746f67067125adb31283c14052f12c.jpg',
     '4MSL0120-175_png.rf.93c8f9d5153f911143651f418f06eacb.jpg',
     '4MSL0119-148_png.rf.e00ecbfdbf3158afb96f78f40ceecfcd.jpg',
     '4MSL0119-113_png.rf.0c51d0505db734bdeae1913e43760ebf.jpg',
     '4MSL0119-67_png.rf.758a0d45240eaa61ad24cf86298f3332.jpg',
     '5MSL3819-48_png.rf.b37d718edf1faf1f279c760d23685211.jpg',
     '5MSL3819-168_png.rf.c791f19bc213d15a6cb4a1c8839261c0.jpg',
     '5MSL0049-23_png.rf.aa4142b069ab92c6eafd7f01633095d5.jpg',
     '5MSL4383-68_png.rf.5360242ffe1ca1a0da6d0062d96169b0.jpg']